In [1]:
import requests
import re
import json
import pandas as pd
import time

In [2]:
pokemon_list = []

for i in range(1, 1026):
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{i}")
    data = response.json()
    species_id = data['species']['url'].split('/')[-2]
    
    species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{species_id}/")
    species_data = species.json()
    
    stats = {s['stat']['name']: s['base_stat'] for s in data['stats']}
    types = [t['type']['name'] for t in data['types']]
    
    pokemon_list.append({
        'id': data['id'],
        'name': data['name'],
        'type1': types[0],
        'type2': types[1] if len(types) > 1 else None,
        'hp': stats['hp'],
        'attack': stats['attack'],
        'defense': stats['defense'],
        'sp_attack': stats['special-attack'],
        'sp_defense': stats['special-defense'],
        'speed': stats['speed'],
        'is_legendary': species_data['is_legendary'],
        'is_mythical': species_data['is_mythical']
    })
    
    time.sleep(0.2)

print(f"Scraped {len(pokemon_list)} base Pokemon")

Scraped 1025 base Pokemon


In [3]:
response = requests.get('https://www.smogon.com/dex/sv/pokemon/')
match = re.search(r'dexSettings = (\{.+\})</script>', response.text, re.DOTALL)
data = json.loads(match.group(1))
basics = data['injectRpcs'][1][1]
pokemon_list_smogon = basics['pokemon']

smogon_data = []

for p in pokemon_list_smogon:
    if p.get('isNonstandard') != 'Standard':
        continue
    if p.get('oob') is None:
        continue
        
    smogon_data.append({
        'name': p['name'],
        'dex_number': p['oob']['dex_number'],
        'tier': p['formats'][0] if p['formats'] else 'Untiered',
        'abilities': ', '.join(p['abilities']),
        'types': ', '.join(p['types']),
        'is_fully_evolved': len(p['oob']['evos']) == 0
    })

df_smogon = pd.DataFrame(smogon_data)
df_smogon['name'] = df_smogon['name'].str.lower().str.replace(' ', '-')
print(f"Scraped {len(df_smogon)} Smogon Pokemon")

Scraped 862 Smogon Pokemon


In [4]:
full_mapping = {
    'tornadus': 'tornadus-incarnate',
    'thundurus': 'thundurus-incarnate',
    'landorus': 'landorus-incarnate',
    'enamorus': 'enamorus-incarnate',
    'minior': 'minior-red-meteor',
    'mimikyu': 'mimikyu-disguised',
    'eiscue': 'eiscue-ice',
    'morpeko': 'morpeko-full-belly',
    'toxtricity': 'toxtricity-amped',
    'urshifu': 'urshifu-single-strike',
    'lycanroc': 'lycanroc-midday',
    'indeedee': 'indeedee-male',
    'indeedee-f': 'indeedee-female',
    'basculin': 'basculin-red-striped',
    'oricorio': 'oricorio-baile',
    "oricorio-pa'u": 'oricorio-pau',
    'meowstic-m': 'meowstic-male',
    'meowstic-f': 'meowstic-female',
    'pyroar': 'pyroar-male',
    'palafin': 'palafin-zero',
    'dudunsparce': 'dudunsparce-two-segment',
    'maushold': 'maushold-family-of-four',
    'maushold-four': 'maushold-family-of-three',
    'squawkabilly': 'squawkabilly-green-plumage',
    'squawkabilly-blue': 'squawkabilly-blue-plumage',
    'squawkabilly-yellow': 'squawkabilly-yellow-plumage',
    'squawkabilly-white': 'squawkabilly-white-plumage',
    'tatsugiri': 'tatsugiri-curly',
    'deoxys': 'deoxys-normal',
    'giratina': 'giratina-altered',
    'shaymin': 'shaymin-land',
    'keldeo': 'keldeo-ordinary',
    'meloetta': 'meloetta-aria',
    'basculegion': 'basculegion-male',
    'basculegion-f': 'basculegion-female',
    'oinkologne': 'oinkologne-male',
    'oinkologne-f': 'oinkologne-female',
    'greninja-bond': 'greninja-battle-bond',
    'zacian-crowned': 'zacian-crowned',
    'zamazenta-crowned': 'zamazenta-crowned',
    'calyrex-ice': 'calyrex-ice',
    'calyrex-shadow': 'calyrex-shadow',
    'necrozma-dawn-wings': 'necrozma-dawn',
    'necrozma-dusk-mane': 'necrozma-dusk',
    'rockruff-dusk': 'rockruff-own-tempo',
    'ogerpon-cornerstone': 'ogerpon-cornerstone-mask',
    'ogerpon-hearthflame': 'ogerpon-hearthflame-mask',
    'ogerpon-wellspring': 'ogerpon-wellspring-mask',
    'tauros-paldea-aqua': 'tauros-paldea-aqua-breed',
    'tauros-paldea-blaze': 'tauros-paldea-blaze-breed',
    'tauros-paldea-combat': 'tauros-paldea-combat-breed',
    'sinistea-antique': 'sinistea',
    'polteageist-antique': 'polteageist',
    'poltchageist-artisan': 'poltchageist',
    'sinistcha-masterpiece': 'sinistcha',
}

df_smogon['name'] = df_smogon['name'].replace(full_mapping)
print("Name standardization complete")

Name standardization complete


In [5]:
df_base = pd.DataFrame(pokemon_list)

unmatched = df_smogon[~df_smogon['name'].isin(df_base['name'])]
alternate_forms = []

for name in unmatched['name'].tolist():
    time.sleep(0.5)
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}")
    if response.status_code == 200:
        data = response.json()
        stats = {s['stat']['name']: s['base_stat'] for s in data['stats']}
        types = [t['type']['name'] for t in data['types']]
        species_id = data['species']['url'].split('/')[-2]
        
        time.sleep(0.5)
        species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{species_id}/")
        species_data = species.json()
        
        alternate_forms.append({
            'id': data['id'],
            'name': name,
            'type1': types[0],
            'type2': types[1] if len(types) > 1 else None,
            'hp': stats['hp'],
            'attack': stats['attack'],
            'defense': stats['defense'],
            'sp_attack': stats['special-attack'],
            'sp_defense': stats['special-defense'],
            'speed': stats['speed'],
            'is_legendary': species_data['is_legendary'],
            'is_mythical': species_data['is_mythical']
        })

print(f"Scraped {len(alternate_forms)} alternate forms")

Scraped 98 alternate forms


In [6]:
arceus_base = df_base[df_base['name'] == 'arceus'].iloc[0]

arceus_types = [
    'bug', 'dark', 'dragon', 'electric', 'fighting', 'fire',
    'flying', 'ghost', 'grass', 'ground', 'ice', 'poison',
    'psychic', 'rock', 'steel', 'water', 'fairy'
]

arceus_forms = []
for t in arceus_types:
    form = arceus_base.to_dict()
    form['name'] = f'arceus-{t}'
    form['type1'] = t
    form['type2'] = None
    arceus_forms.append(form)

print(f"Created {len(arceus_forms)} Arceus forms")

Created 17 Arceus forms


In [7]:
df_alternate = pd.DataFrame(alternate_forms)
df_arceus = pd.DataFrame(arceus_forms)

# Combine all Pokemon data
df_stats = pd.concat([df_base, df_alternate, df_arceus], ignore_index=True)
df_stats = df_stats.drop_duplicates(subset='name', keep='first')

# Deduplicate Smogon data
df_smogon = df_smogon.drop_duplicates(subset='name', keep='first')

# Final merge
df_merged = pd.merge(df_stats, df_smogon, on='name', how='inner')
df_merged = df_merged.drop_duplicates(subset='name', keep='first')

# Save all three
df_stats.to_csv('pokemon_data.csv', index=False)
df_smogon.to_csv('smogon_data.csv', index=False)
df_merged.to_csv('merged_data.csv', index=False)

print(f"pokemon_data.csv: {len(df_stats)} Pokemon")
print(f"smogon_data.csv: {len(df_smogon)} Pokemon")
print(f"merged_data.csv: {len(df_merged)} Pokemon")

pokemon_data.csv: 1140 Pokemon
smogon_data.csv: 858 Pokemon
merged_data.csv: 848 Pokemon
